# Week 3a.2 — Structured Output

An agent's answer is prose. That is fine when a human reads it, and useless when a program does: if the answer feeds a database, a UI, or another agent, you need fields with types, not a paragraph. Today: making the model return data.

In [ ]:
from dotenv import load_dotenv
import os
import logging

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."

# silence a noisy advisory warning from the Google SDK
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. The problem

In [ ]:
email = """Subject: STILL waiting

I ordered a mechanical keyboard two weeks ago, order HT-1002, and it has still
not arrived. This is the third time I am writing. If it is not here by Friday
I want my money back."""

response = model.invoke(f"Extract the order id and the problem from this email:\n\n{email}")
print(response.text)

A perfectly good answer for a human, but try writing code against it. The phrasing changes run to run; string parsing breaks with it. We need to tell the model the exact shape we want.

## 2. Describing the shape: Pydantic

We describe the shape as a **Pydantic model**: a class listing each field with a type and a description. The descriptions are not comments — like tool docstrings, the model reads them to decide what goes where. `Literal` restricts a field to fixed choices.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class SupportTicket(BaseModel):
    """A structured record of one customer support request."""
    order_id: str = Field(description="The order id mentioned, e.g. HT-1001")
    category: Literal["billing", "shipping", "returns", "other"] = Field(
        description="What kind of problem this is")
    summary: str = Field(description="The problem in one sentence")
    urgent: bool = Field(description="Whether the customer needs an immediate response")

## 3. `with_structured_output`

`with_structured_output` wraps the model so it must answer in that shape. Under the hood this is tool calling: the schema is handed to the model as the one tool it has to call.

- https://docs.langchain.com/oss/python/langchain/structured-output

In [ ]:
structured_model = model.with_structured_output(SupportTicket)

ticket = structured_model.invoke(f"Extract a support ticket from this email:\n\n{email}")
ticket

In [ ]:
print(ticket.order_id)
print(ticket.category)
print(ticket.urgent)
print(type(ticket))

Not a string that looks like data — an actual Python object. `ticket.order_id` is a `str`, `ticket.urgent` is a `bool`, and `category` is guaranteed to be one of the four allowed values.

In [ ]:
#TODO: define a ProductReview model and extract one from the text below.
# Fields: rating (int, 1 to 5), pros (list[str]), cons (list[str]).
# Give every field a description.

review = """These headphones sound amazing for the price and the battery lasts
all week. The ear cushions get sweaty after an hour though, and the app is
useless. Solid 4 out of 5 from me."""


## 4. Structured output from agents

Agents support the same idea: pass `response_format` to `create_agent` and the final answer arrives twice — the prose reply in `messages`, and the typed object under `structured_response`. Tools still work along the way.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You turn customer emails into support tickets.",
    response_format=SupportTicket,
)

result = agent.invoke({"messages": [HumanMessage(content=email)]})
result["structured_response"]

## 5. ICA: a triage pipeline

Three emails arrived overnight. Extract a `SupportTicket` from each and print the urgent ones first. This is the shape of real intake automation: unstructured text in, sorted queue out.

In [ ]:
emails = [
    "Order HT-1001: the headphones arrived cracked. I want to send them back for a refund.",
    "Hey, quick question, does the USB-C dock in order HT-1003 work with a MacBook?",
    "You charged my card twice for order HT-1002!! Fix this today or I dispute the charge.",
]

#TODO: loop over the emails, extract a SupportTicket from each,
# and print urgent tickets before non-urgent ones.


On Friday, everything from today comes together: an agent with tools, memory per customer, and a structured ticket at the end of the conversation.